In [1]:
from pathlib import Path
import pandas as pd
import duckdb

# ------------------------------------------------------------
# PART 1: LOCATE PROJECT AND VALIDATED WAREHOUSE
# ------------------------------------------------------------

project_root = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (
            p / "config" / "signals_v2"
            / "table_manifest.csv"
        ).is_file()
    ),
    None
)

assert project_root is not None, "Project root not found."

database_path = (
    project_root
    / "data"
    / "warehouse"
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

assert database_path.is_file()

print("===== EXECUTIVE DASHBOARD CONFIGURATION =====")
print("Project:", project_root.name)
print("Warehouse:", database_path.name)
print("Connection mode: READ ONLY")


# ------------------------------------------------------------
# PART 2: DEFINE DASHBOARD BUSINESS AREAS
# ------------------------------------------------------------

def assign_dashboard_area(view_name):

    name = view_name.lower()

    if "sensitivity" in name:
        return "Sensitivity & Decision Intelligence"

    if "inventory" in name:
        return "Inventory & Launch Readiness"

    if (
        "forecast" in name
        or "airpods" in name
        or "external_signal" in name
    ):
        return "Demand & Forecast Intelligence"

    if "retail" in name or "headphone" in name:
        return "Market & Retail Context"

    if "purchase_intent" in name:
        return "Customer Intent Research"

    return "Other"


# ------------------------------------------------------------
# PART 3: INSPECT SQL REPORTING VIEWS
# ------------------------------------------------------------

def quote_identifier(value):
    return '"' + str(value).replace('"', '""') + '"'


con = duckdb.connect(
    str(database_path),
    read_only=True
)

try:

    view_names = con.execute(
        """
        SELECT view_name
        FROM duckdb_views()
        WHERE schema_name = 'reporting'
        ORDER BY view_name
        """
    ).fetchall()

    assert len(view_names) == 18

    records = []

    for (view_name,) in view_names:

        qualified_name = (
            "reporting."
            + quote_identifier(view_name)
        )

        row_count = con.execute(
            f"SELECT COUNT(*) FROM {qualified_name}"
        ).fetchone()[0]

        description = con.execute(
            f"DESCRIBE SELECT * FROM {qualified_name}"
        ).fetchall()

        column_names = [
            row[0] for row in description
        ]

        date_fields = [
            column
            for column in column_names
            if any(
                word in column.lower()
                for word in [
                    "date",
                    "week",
                    "month",
                    "period"
                ]
            )
        ]

        records.append({
            "Reporting_View": view_name,
            "Dashboard_Area": assign_dashboard_area(
                view_name
            ),
            "Rows": row_count,
            "Columns": len(column_names),
            "Date_Fields": ", ".join(date_fields),
            "Available_Fields": ", ".join(column_names)
        })

    dashboard_source_inventory = pd.DataFrame(
        records
    )

    print("\n===== REPORTING SOURCE INVENTORY =====")

    display(
        dashboard_source_inventory[
            [
                "Reporting_View",
                "Dashboard_Area",
                "Rows",
                "Columns"
            ]
        ]
    )


    # --------------------------------------------------------
    # PART 4: SHOW FIELD STRUCTURES
    # --------------------------------------------------------

    print("\n===== DASHBOARD FIELD DICTIONARY =====")

    display(
        dashboard_source_inventory[
            [
                "Reporting_View",
                "Date_Fields",
                "Available_Fields"
            ]
        ]
    )


    # --------------------------------------------------------
    # PART 5: SUMMARIZE BUSINESS AREAS
    # --------------------------------------------------------

    area_summary = (
        dashboard_source_inventory
        .groupby("Dashboard_Area")
        .agg(
            Reporting_Views=("Reporting_View", "count"),
            Total_Rows=("Rows", "sum")
        )
        .reset_index()
        .sort_values("Dashboard_Area")
    )

    print("\n===== DASHBOARD BUSINESS AREA SUMMARY =====")

    display(area_summary)


    # --------------------------------------------------------
    # PART 6: VERIFY HISTORICAL BUSINESS BENCHMARKS
    # --------------------------------------------------------

    historical_units = con.execute(
        """
        SELECT SUM(Units_Sold)
        FROM reporting.v_airpods_weekly_demand
        """
    ).fetchone()[0]

    historical_weeks = con.execute(
        """
        SELECT COUNT(*)
        FROM reporting.v_airpods_weekly_demand
        """
    ).fetchone()[0]

    print("\n===== HISTORICAL DEMAND BENCHMARK =====")
    print("AirPods units:", historical_units)
    print("Historical weeks:", historical_weeks)

    assert historical_units == 15364
    assert historical_weeks == 51


    # --------------------------------------------------------
    # PART 7: VALIDATE DASHBOARD INVENTORY
    # --------------------------------------------------------

    assert len(dashboard_source_inventory) == 18

    assert (
        dashboard_source_inventory["Reporting_View"]
        .is_unique
    )

    assert (
        dashboard_source_inventory["Columns"] > 0
    ).all()

    print("\n===== FINAL DASHBOARD SOURCE VALIDATION =====")

    print("18 reporting views inspected: PASS")
    print("Unique reporting view names: PASS")
    print("All views have defined columns: PASS")
    print("Historical AirPods demand reconciled: PASS")
    print("Read-only warehouse connection: PASS")

    print("\n===== STEP 1 COMPLETE =====")

finally:

    con.close()
    print("\nDatabase connection closed safely.")


===== EXECUTIVE DASHBOARD CONFIGURATION =====
Project: hardware-demand-sensing-planner
Warehouse: demand_intelligence_signals_manifest_rebuild.duckdb
Connection mode: READ ONLY

===== REPORTING SOURCE INVENTORY =====


,Reporting_View,Dashboard_Area,Rows,Columns
0,v_airpods_weekly_demand,Demand & Forecast Intelligence,51,2
1,v_external_signal_aligned_demand,Demand & Forecast Intelligence,51,6
2,v_external_signal_model_comparison,Demand & Forecast Intelligence,3,7
3,v_forecast_lead_time_baseline,Demand & Forecast Intelligence,4,7
4,v_forecast_model_performance,Demand & Forecast Intelligence,8,6
5,v_forecast_multihorizon_performance,Demand & Forecast Intelligence,15,7
6,v_headphone_daily_signals,Market & Retail Context,61,7
7,v_inventory_candidate_actions,Inventory & Launch Readiness,2,5
8,v_inventory_launch_readiness,Inventory & Launch Readiness,7,9
9,v_inventory_scenario_performance,Inventory & Launch Readiness,3,8



===== DASHBOARD FIELD DICTIONARY =====


,Reporting_View,Date_Fields,Available_Fields
0,v_airpods_weekly_demand,Week_Ending,"Week_Ending, Units_Sold"
1,v_external_signal_aligned_demand,"Week_Ending, Overlapping_Week_Search_Interest,...","Week_Ending, Units_Sold, Forecast_Origin, Over..."
2,v_external_signal_model_comparison,,"Model, Forecasts, MAE, WAPE_Pct, Bias, WAPE_Ga..."
3,v_forecast_lead_time_baseline,"Forecast_Week, Lead_Time_Weeks, Forecast_Weekl...","Forecast_Origin, Forecast_Week, Lead_Time_Week..."
4,v_forecast_model_performance,,"Experiment, Target_Unit, Model, MAE, WAPE_Pct,..."
5,v_forecast_multihorizon_performance,Horizon_Weeks,"Horizon_Weeks, Model, Forecasts, MAE, WAPE_Pct..."
6,v_headphone_daily_signals,Event_Date,"Event_Date, Views, Cart_Events, Recorded_Purch..."
7,v_inventory_candidate_actions,,"Scenario, Action, Fill_Rate_Pct, Illustrative_..."
8,v_inventory_launch_readiness,Shortage_Weeks,"Scenario, Action, Total_Shortage, Shortage_Wee..."
9,v_inventory_scenario_performance,Shortage_Weeks,"Scenario, Total_Demand, Total_Fulfilled, Total..."



===== DASHBOARD BUSINESS AREA SUMMARY =====


,Dashboard_Area,Reporting_Views,Total_Rows
0,Customer Intent Research,1,7
1,Demand & Forecast Intelligence,6,132
2,Inventory & Launch Readiness,4,24
3,Market & Retail Context,2,123
4,Sensitivity & Decision Intelligence,5,744



===== HISTORICAL DEMAND BENCHMARK =====
AirPods units: 15364.0
Historical weeks: 51

===== FINAL DASHBOARD SOURCE VALIDATION =====
18 reporting views inspected: PASS
Unique reporting view names: PASS
All views have defined columns: PASS
Historical AirPods demand reconciled: PASS
Read-only warehouse connection: PASS

===== STEP 1 COMPLETE =====

Database connection closed safely.


In [ ]:
from pathlib import Path
import pandas as pd
import duckdb

# PART 1: OUTPUT DIRECTORIES

dashboard_dir = (
    project_root / "data" / "dashboard" / "executive_v1"
)

validation_dir = (
    project_root / "reports" / "validation"
)

dashboard_dir.mkdir(parents=True, exist_ok=True)
validation_dir.mkdir(parents=True, exist_ok=True)

print("===== DASHBOARD EXPORT CONFIGURATION =====")
print("Output:", dashboard_dir)
print("Source:", database_path.name)

# PART 2: CONNECT TO VALIDATED DATABASE

con = duckdb.connect(
    str(database_path),
    read_only=True
)

try:

    views = con.execute("""
        SELECT view_name
        FROM duckdb_views()
        WHERE schema_name = 'reporting'
        ORDER BY view_name
    """).fetchall()

    assert len(views) == 18

    export_records = []
    field_records = []

    print("\n===== EXPORTING REPORTING VIEWS =====")

    for (view_name,) in views:

        assert view_name.replace("_", "").isalnum()

        query = f'SELECT * FROM reporting."{view_name}"'

        df = con.execute(query).fetchdf()

        schema_info = con.execute(
            f"DESCRIBE {query}"
        ).fetchall()

        filename = f"dashboard_{view_name}.csv"
        output_path = dashboard_dir / filename

        df.to_csv(output_path, index=False)

        # Validate written CSV
        reloaded = pd.read_csv(output_path)

        assert reloaded.shape == df.shape
        assert reloaded.columns.tolist() == df.columns.tolist()

        export_records.append({
            "Reporting_View": view_name,
            "File_Name": filename,
            "Rows": len(df),
            "Columns": len(df.columns),
            "Status": "PASS"
        })

        for position, column in enumerate(schema_info, 1):
            field_records.append({
                "Reporting_View": view_name,
                "Column_Position": position,
                "Column_Name": column[0],
                "Data_Type": column[1]
            })

        print(
            f"PASS: {filename} "
            f"({len(df)} rows)"
        )

    # PART 3: BUILD DATA CATALOGS

    dashboard_export_manifest = pd.DataFrame(
        export_records
    )

    dashboard_field_catalog = pd.DataFrame(
        field_records
    )

    dashboard_export_manifest.to_csv(
        dashboard_dir / "dashboard_export_manifest.csv",
        index=False
    )

    dashboard_field_catalog.to_csv(
        dashboard_dir / "dashboard_field_catalog.csv",
        index=False
    )

    # PART 4: SAVE AUDIT REPORT

    dashboard_export_manifest.to_csv(
        validation_dir / "dashboard_export_validation.csv",
        index=False
    )

    print("\n===== DASHBOARD EXPORT SUMMARY =====")
    display(dashboard_export_manifest)

    print("\n===== COLUMN DICTIONARY SUMMARY =====")
    print(
        "Total documented fields:",
        len(dashboard_field_catalog)
    )

    # PART 5: FINAL VALIDATION

    assert len(dashboard_export_manifest) == 18
    assert dashboard_export_manifest["Status"].eq("PASS").all()

    assert dashboard_export_manifest[
        "Reporting_View"
    ].is_unique

    assert len(dashboard_field_catalog) > 0

    print("\n===== FINAL VALIDATION =====")
    print("18 reporting views exported: PASS")
    print("18 CSV structure checks: PASS")
    print("Export manifest created: PASS")
    print("Column dictionary created: PASS")
    print("Validation report saved: PASS")

    print("\n===== STEP 2 COMPLETE =====")

finally:
    con.close()
    print("\nDatabase connection closed safely.")
